# ROUTING

## ENVIRONMENT

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
if os.getenv("LANGCHAIN_API_KEY"):
    os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
if os.getenv("OPENAI_API_KEY"):
    os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

#### LOGICAL ROUTING

LLM reads the question and decides with reasoning.

In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

from langchain_core.runnables import RunnableLambda

In [ ]:
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )

Here we are defining the output format for llm.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

Normal LLM returns plain text this makes it return a structured object.

In [ ]:

system = """You are an expert at routing a user question to the appropriate data source.
Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

router = prompt | structured_llm

Here we are providing the prompt and router return which type of datasource to use.

In [ ]:
question = """Why doesn't the following code work:
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})

In this block we ask question and use router to find the datasource.

In [ ]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        return "chain for js_docs"
    else:
        return "golang_docs"
    
full_chain = router | RunnableLambda(choose_route)
full_chain.invoke({"question": question})

Once we find the data source we print it using the function choose_route.

## 🔄 Logical Routing Flowchart

Below is the flow of the Logical Routing system, structured in a compact layout with clear visibility:

<br>

![Logical Routing Flowchart](https://mermaid.ink/img/Zmxvd2NoYXJ0IFRCCiAgICAlJSBOb2RlcyAmIFN0eWxpbmcKICAgIHN1YmdyYXBoIExvZ2ljYWxSb3V0aW5nIFsi8J+UgCBMT0dJQ0FMIFJPVVRJTkcgRkxPVyA8YnI+ICJdCiAgICAgICAgZGlyZWN0aW9uIFRCCiAgICAgICAgc3ViZ3JhcGggUm93MSBbIklucHV0ICYgQW5hbHlzaXMiXQogICAgICAgICAgICBkaXJlY3Rpb24gTFIKICAgICAgICAgICAgUXVlcnlbIvCfkqwgVXNlciBRdWVzdGlvbiJdIC0tPiBSZWFkWyLwn6egIExMTSBSZWFkcyJdCiAgICAgICAgZW5kCiAgICAgICAgc3ViZ3JhcGggUm93MiBbIlJvdXRpbmcgJiBSZXRyaWV2YWwiXQogICAgICAgICAgICBkaXJlY3Rpb24gTFIKICAgICAgICAgICAgUm91dGVbIuKame+4jyBTZWxlY3QgRGF0YXNvdXJjZSJdIC0tPiBDaGFpblsi8J+UlyBSdW4gUkFHIENoYWluIl0KICAgICAgICBlbmQKICAgICAgICBSb3cxIC0tPiBSb3cyCiAgICAgICAgUm93MiAtLT4gQW5zd2VyWyLinInvuI8gRmluYWwgQW5zd2VyIl0KICAgIGVuZAoKICAgICUlIENvbG9ycyAmIEFlc3RoZXRpY3MKICAgIHN0eWxlIExvZ2ljYWxSb3V0aW5nIGZpbGw6I2Y4ZmFmYyxzdHJva2U6Izk0YTNiOCxzdHJva2Utd2lkdGg6MXB4LHN0cm9rZS1kYXNoYXJyYXk6IDUgNTsKICAgIHN0eWxlIFJvdzEgZmlsbDojZjFmNWY5LHN0cm9rZTojY2JkNWUxLHN0cm9rZS13aWR0aDoxcHg7CiAgICBzdHlsZSBSb3cyIGZpbGw6I2YxZjVmOSxzdHJva2U6I2NiZDVlMSxzdHJva2Utd2lkdGg6MXB4OwogICAgCiAgICBjbGFzc0RlZiBzdGVwIGZpbGw6I2ZmZmZmZixzdHJva2U6IzNiODJmNixzdHJva2Utd2lkdGg6MnB4LGNvbG9yOiMxZTNhOGEsZm9udC13ZWlnaHQ6Ym9sZDsKICAgIGNsYXNzRGVmIHF1ZXJ5IGZpbGw6I2ZlZjNjNyxzdHJva2U6I2Q5NzcwNixzdHJva2Utd2lkdGg6MnB4LGNvbG9yOiM3ODM1MGYsZm9udC13ZWlnaHQ6Ym9sZDsKICAgIGNsYXNzRGVmIHNlbGVjdCBmaWxsOiNmZWY2ZmYsc3Ryb2tlOiMyNTYzZWIsc3Ryb2tlLXdpZHRoOjJweCxjb2xvcjojMWUzYThhLGZvbnQtd2VpZ2h0OmJvbGQ7CiAgICBjbGFzc0RlZiBhbnN3ZXIgZmlsbDojZWNmZGY1LHN0cm9rZTojMDU5NjY5LHN0cm9rZS13aWR0aDoycHgsY29sb3I6IzA2NWY0Nixmb250LXdlaWdodDpib2xkOwoKICAgIGNsYXNzIFJlYWQsQ2hhaW4gc3RlcDsKICAgIGNsYXNzIFF1ZXJ5IHF1ZXJ5OwogICAgY2xhc3MgUm91dGUgc2VsZWN0OwogICAgY2xhc3MgQW5zd2VyIGFuc3dlcjs=)

<details>
<summary>Click to view raw Mermaid diagram source</summary>

```mermaid
flowchart TB
    %% Nodes & Styling
    subgraph LogicalRouting ["🔀 LOGICAL ROUTING FLOW <br> "]
        direction TB
        subgraph Row1 ["Input & Analysis"]
            direction LR
            Query["💬 User Question"] --> Read["🧠 LLM Reads"]
        end
        subgraph Row2 ["Routing & Retrieval"]
            direction LR
            Route["⚙️ Select Datasource"] --> Chain["🔗 Run RAG Chain"]
        end
        Row1 --> Row2
        Row2 --> Answer["✉️ Final Answer"]
    end

    %% Colors & Aesthetics
    style LogicalRouting fill:#f8fafc,stroke:#94a3b8,stroke-width:1px,stroke-dasharray: 5 5;
    style Row1 fill:#f1f5f9,stroke:#cbd5e1,stroke-width:1px;
    style Row2 fill:#f1f5f9,stroke:#cbd5e1,stroke-width:1px;
    
    classDef step fill:#ffffff,stroke:#3b82f6,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef query fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f,font-weight:bold;
    classDef select fill:#fef6ff,stroke:#2563eb,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef answer fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#065f46,font-weight:bold;

    class Read,Chain step;
    class Query query;
    class Route select;
    class Answer answer;
```
</details>

#### SEMENTIC ROUTING

Embeddings decide based on similarity. No LLM needed for routing.

In [ ]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

Here we are giving two prompt and it will decide according to question.

In [ ]:
embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

Here we form embedding for both prompt.

In [ ]:
def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

In this block of code we find the cosine similarity between question and both the prompt the most similar prompt will use.

In [ ]:
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatOpenAI()
    | StrOutputParser()
)

print(chain.invoke("What's a black hole"))

This is the final stepp were we connect all and hense find both datasource and answer to the query.

## 🔄 Semantic Routing Flowchart

Below is the flow of the Semantic Routing system, structured in a compact layout with clear visibility:

<br>

![Semantic Routing Flowchart](https://mermaid.ink/img/Zmxvd2NoYXJ0IFRCCiAgICAlJSBOb2RlcyAmIFN0eWxpbmcKICAgIHN1YmdyYXBoIFNlbWFudGljUm91dGluZyBbIvCfp6AgU0VNQU5USUMgUk9VVElORyBGTE9XIDxicj4gIl0KICAgICAgICBkaXJlY3Rpb24gVEIKICAgICAgICBzdWJncmFwaCBSb3cxIFsiMS4gRW1iZWQgJiBDb21wYXJlIl0KICAgICAgICAgICAgZGlyZWN0aW9uIExSCiAgICAgICAgICAgIFF1ZXJ5WyLwn5KsIFVzZXIgUXVlc3Rpb24iXSAtLT4gRW1iZWRbIvCfp6AgRW1iZWQgUXVlc3Rpb24iXSAtLT4gQ29tcGFyZVsi8J+UjSBDb3NpbmUgU2ltaWxhcml0eSJdCiAgICAgICAgZW5kCiAgICAgICAgc3ViZ3JhcGggUm93MiBbIjIuIFNlbGVjdCAmIEdlbmVyYXRlIl0KICAgICAgICAgICAgZGlyZWN0aW9uIExSCiAgICAgICAgICAgIFNlbGVjdFsi8J+OryBTZWxlY3QgUHJvbXB0Il0gLS0+IExMTVsi8J+kliBMTE0gQW5zd2VycyJdIC0tPiBBbnN3ZXJbIuKcie+4jyBGaW5hbCBBbnN3ZXIiXQogICAgICAgIGVuZAogICAgICAgIFJvdzEgLS0+IFJvdzIKICAgIGVuZAoKICAgICUlIENvbG9ycyAmIEFlc3RoZXRpY3MKICAgIHN0eWxlIFNlbWFudGljUm91dGluZyBmaWxsOiNmOGZhZmMsc3Ryb2tlOiM5NGEzYjgsc3Ryb2tlLXdpZHRoOjFweCxzdHJva2UtZGFzaGFycmF5OiA1IDU7CiAgICBzdHlsZSBSb3cxIGZpbGw6I2YxZjVmOSxzdHJva2U6I2NiZDVlMSxzdHJva2Utd2lkdGg6MXB4OwogICAgc3R5bGUgUm93MiBmaWxsOiNmMWY1Zjksc3Ryb2tlOiNjYmQ1ZTEsc3Ryb2tlLXdpZHRoOjFweDsKICAgIAogICAgY2xhc3NEZWYgc3RlcCBmaWxsOiNmZmZmZmYsc3Ryb2tlOiMzYjgyZjYsc3Ryb2tlLXdpZHRoOjJweCxjb2xvcjojMWUzYThhLGZvbnQtd2VpZ2h0OmJvbGQ7CiAgICBjbGFzc0RlZiBxdWVyeSBmaWxsOiNmZWYzYzcsc3Ryb2tlOiNkOTc3MDYsc3Ryb2tlLXdpZHRoOjJweCxjb2xvcjojNzgzNTBmLGZvbnQtd2VpZ2h0OmJvbGQ7CiAgICBjbGFzc0RlZiBzZWxlY3QgZmlsbDojZmVmNmZmLHN0cm9rZTojMjU2M2ViLHN0cm9rZS13aWR0aDoycHgsY29sb3I6IzFlM2E4YSxmb250LXdlaWdodDpib2xkOwogICAgY2xhc3NEZWYgYW5zd2VyIGZpbGw6I2VjZmRmNSxzdHJva2U6IzA1OTY2OSxzdHJva2Utd2lkdGg6MnB4LGNvbG9yOiMwNjVmNDYsZm9udC13ZWlnaHQ6Ym9sZDsKCiAgICBjbGFzcyBFbWJlZCxDb21wYXJlLExMTSBzdGVwOwogICAgY2xhc3MgUXVlcnkgcXVlcnk7CiAgICBjbGFzcyBTZWxlY3Qgc2VsZWN0OwogICAgY2xhc3MgQW5zd2VyIGFuc3dlcjs=)

<details>
<summary>Click to view raw Mermaid diagram source</summary>

```mermaid
flowchart TB
    %% Nodes & Styling
    subgraph SemanticRouting ["🧠 SEMANTIC ROUTING FLOW <br> "]
        direction TB
        subgraph Row1 ["1. Embed & Compare"]
            direction LR
            Query["💬 User Question"] --> Embed["🧠 Embed Question"] --> Compare["🔍 Cosine Similarity"]
        end
        subgraph Row2 ["2. Select & Generate"]
            direction LR
            Select["🎯 Select Prompt"] --> LLM["🤖 LLM Answers"] --> Answer["✉️ Final Answer"]
        end
        Row1 --> Row2
    end

    %% Colors & Aesthetics
    style SemanticRouting fill:#f8fafc,stroke:#94a3b8,stroke-width:1px,stroke-dasharray: 5 5;
    style Row1 fill:#f1f5f9,stroke:#cbd5e1,stroke-width:1px;
    style Row2 fill:#f1f5f9,stroke:#cbd5e1,stroke-width:1px;
    
    classDef step fill:#ffffff,stroke:#3b82f6,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef query fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f,font-weight:bold;
    classDef select fill:#fef6ff,stroke:#2563eb,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef answer fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#065f46,font-weight:bold;

    class Embed,Compare,LLM step;
    class Query query;
    class Select select;
    class Answer answer;
```
</details>

# QUERY CONSTRUCTION

 It convert a natural language question into a structured database query with filters

In [ ]:
from langchain_community.document_loaders import YoutubeLoader

docs = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=pbAd8O1Lvm4", add_video_info=True
).load()

docs[0].metadata

Here we give te youtube video database.

In [ ]:
import datetime
from typing import Literal, Optional, Tuple
from pydantic import BaseModel, Field

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    content_search: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )
    title_search: str = Field(
        ...,
        description=(
            "Alternate version of the content search query to apply to video titles. "
            "Should be succinct and only include key words that could be in a video "
            "title."
        ),
    )
    min_view_count: Optional[int] = Field(
        None,
        description="Minimum view count filter, inclusive. Only use if explicitly specified.",
    )
    max_view_count: Optional[int] = Field(
        None,
        description="Maximum view count filter, exclusive. Only use if explicitly specified.",
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Earliest publish date filter, inclusive. Only use if explicitly specified.",
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Latest publish date filter, exclusive. Only use if explicitly specified.",
    )
    min_length_sec: Optional[int] = Field(
        None,
        description="Minimum video length in seconds, inclusive. Only use if explicitly specified.",
    )
    max_length_sec: Optional[int] = Field(
        None,
        description="Maximum video length in seconds, exclusive. Only use if explicitly specified.",
    )
    
    def pretty_print(self) -> None:
        for field_name, field_info in self.__class__.model_fields.items():
            value = getattr(self, field_name)
            if value is not None and value != field_info.default:
                print(f"{field_name}: {value}")

Here we define the class. This is like a form that llm must fill.Some are optional to fill.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(TutorialSearch)
query_analyzer = prompt | structured_llm


Here we define the system and then query is asked in simple English  .LLM read that query and fill the tutorial search schema, and then it finally prepared a structured query with filters.

In [ ]:
query_analyzer.invoke(
    {
        "question": "how to use multi-modal models in an agent, only videos under 5 minutes"
    }
).pretty_print()

In [ ]:
query_analyzer.invoke(
    {"question": "videos that are focused on the topic of chat langchain that are published before 2024"}
).pretty_print()

In [ ]:
query_analyzer.invoke(
    {"question": "videos on chat langchain published in 2023"}
).pretty_print()

## 🔄 Query Construction Flowchart

Below is the flow of the Query Construction system, structured in a compact layout with clear visibility:

<br>

![Query Construction Flowchart](https://mermaid.ink/img/Zmxvd2NoYXJ0IFRCCiAgICAlJSBOb2RlcyAmIFN0eWxpbmcKICAgIHN1YmdyYXBoIFF1ZXJ5Q29uc3RydWN0aW9uIFsi8J+UjSBRVUVSWSBDT05TVFJVQ1RJT04gRkxPVyA8YnI+ICJdCiAgICAgICAgZGlyZWN0aW9uIFRCCiAgICAgICAgc3ViZ3JhcGggUm93MSBbIjEuIElucHV0ICYgQW5hbHlzaXMiXQogICAgICAgICAgICBkaXJlY3Rpb24gTFIKICAgICAgICAgICAgUXVlcnlbIvCfkqwgVXNlciBRdWVzdGlvbiJdIC0tPiBSZWFkWyLwn6egIExMTSBSZWFkcyJdIC0tPiBTY2hlbWFbIvCfk4sgRmlsbHMgU2NoZW1hIl0KICAgICAgICBlbmQKICAgICAgICBzdWJncmFwaCBSb3cyIFsiMi4gUXVlcnkgJiBTZWFyY2giXQogICAgICAgICAgICBkaXJlY3Rpb24gTFIKICAgICAgICAgICAgU3RydWN0dXJlZFsi8J+UjSBTdHJ1Y3R1cmVkIFF1ZXJ5Il0gLS0+IFNlYXJjaFsi8J+XhO+4jyBEYXRhYmFzZSBTZWFyY2giXSAtLT4gUmVzdWx0c1si8J+OryBQcmVjaXNlIFJlc3VsdHMiXQogICAgICAgIGVuZAogICAgICAgIFJvdzEgLS0+IFJvdzIKICAgIGVuZAoKICAgICUlIENvbG9ycyAmIEFlc3RoZXRpY3MKICAgIHN0eWxlIFF1ZXJ5Q29uc3RydWN0aW9uIGZpbGw6I2Y4ZmFmYyxzdHJva2U6Izk0YTNiOCxzdHJva2Utd2lkdGg6MXB4LHN0cm9rZS1kYXNoYXJyYXk6IDUgNTsKICAgIHN0eWxlIFJvdzEgZmlsbDojZjFmNWY5LHN0cm9rZTojY2JkNWUxLHN0cm9rZS13aWR0aDoxcHg7CiAgICBzdHlsZSBSb3cyIGZpbGw6I2YxZjVmOSxzdHJva2U6I2NiZDVlMSxzdHJva2Utd2lkdGg6MXB4OwogICAgCiAgICBjbGFzc0RlZiBzdGVwIGZpbGw6I2ZmZmZmZixzdHJva2U6IzNiODJmNixzdHJva2Utd2lkdGg6MnB4LGNvbG9yOiMxZTNhOGEsZm9udC13ZWlnaHQ6Ym9sZDsKICAgIGNsYXNzRGVmIHF1ZXJ5IGZpbGw6I2ZlZjNjNyxzdHJva2U6I2Q5NzcwNixzdHJva2Utd2lkdGg6MnB4LGNvbG9yOiM3ODM1MGYsZm9udC13ZWlnaHQ6Ym9sZDsKICAgIGNsYXNzRGVmIHNlbGVjdCBmaWxsOiNmZWY2ZmYsc3Ryb2tlOiMyNTYzZWIsc3Ryb2tlLXdpZHRoOjJweCxjb2xvcjojMWUzYThhLGZvbnQtd2VpZ2h0OmJvbGQ7CiAgICBjbGFzc0RlZiBhbnN3ZXIgZmlsbDojZWNmZGY1LHN0cm9rZTojMDU5NjY5LHN0cm9rZS13aWR0aDoycHgsY29sb3I6IzA2NWY0Nixmb250LXdlaWdodDpib2xkOwoKICAgIGNsYXNzIFJlYWQsU2NoZW1hLFNlYXJjaCBzdGVwOwogICAgY2xhc3MgUXVlcnkgcXVlcnk7CiAgICBjbGFzcyBTdHJ1Y3R1cmVkIHNlbGVjdDsKICAgIGNsYXNzIFJlc3VsdHMgYW5zd2VyOw==)

<details>
<summary>Click to view raw Mermaid diagram source</summary>

```mermaid
flowchart TB
    %% Nodes & Styling
    subgraph QueryConstruction ["🔍 QUERY CONSTRUCTION FLOW <br> "]
        direction TB
        subgraph Row1 ["1. Input & Analysis"]
            direction LR
            Query["💬 User Question"] --> Read["🧠 LLM Reads"] --> Schema["📋 Fills Schema"]
        end
        subgraph Row2 ["2. Query & Search"]
            direction LR
            Structured["🔍 Structured Query"] --> Search["🗄️ Database Search"] --> Results["🎯 Precise Results"]
        end
        Row1 --> Row2
    end

    %% Colors & Aesthetics
    style QueryConstruction fill:#f8fafc,stroke:#94a3b8,stroke-width:1px,stroke-dasharray: 5 5;
    style Row1 fill:#f1f5f9,stroke:#cbd5e1,stroke-width:1px;
    style Row2 fill:#f1f5f9,stroke:#cbd5e1,stroke-width:1px;
    
    classDef step fill:#ffffff,stroke:#3b82f6,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef query fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f,font-weight:bold;
    classDef select fill:#fef6ff,stroke:#2563eb,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef answer fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#065f46,font-weight:bold;

    class Read,Schema,Search step;
    class Query query;
    class Structured select;
    class Results answer;
```
</details>